In [21]:
import pandas as pd
import numpy, sklearn, streamlit, matplotlib

import matplotlib.pyplot as plt
from pathlib import Path

# Directori de dades (detecta tant si s'executa des de starter_kit/ com des de l'arrel)
DATA_DIR = Path("../data/train") if Path("../data/train").exists() else Path("data/train")

# Carreguem les dades principals
df_contador = pd.read_csv(DATA_DIR / "contador.csv", parse_dates=["Fecha"])
df_inversor = pd.read_csv(DATA_DIR / "inversor.csv", parse_dates=["Fecha"])
df_meteo = pd.read_csv(DATA_DIR / "meteo.csv", parse_dates=["Fecha"])
df_pvpc = pd.read_csv(DATA_DIR / "precio_pvpc.csv", parse_dates=["Fecha"])
df_co2 = pd.read_csv(DATA_DIR / "co2_red.csv", parse_dates=["Fecha"])

In [22]:
# 1. Consum del campus (kW)
df_contador = df_contador.set_index("Fecha")
load_5min = (-df_contador["Contador - POTENCIA ACTIVA EXPORTADA (kW)"]).clip(lower=0, upper=5000)
load_hourly = load_5min.resample("1h").mean().dropna()


# 2. Producció solar fotovoltaica total (kW)
df_inversor = df_inversor.set_index("Fecha")
p1 = [c for c in df_inversor.columns if "INVERSOR 1" in c.upper() and "POTENCIA ACTIVA" in c.upper() and "FASE" not in c.upper()][0]
p2 = [c for c in df_inversor.columns if "INVERSOR 2" in c.upper() and "POTENCIA ACTIVA" in c.upper() and "FASE" not in c.upper()][0]
p3 = [c for c in df_inversor.columns if "INVERSOR 3" in c.upper() and "POTENCIA ACTIVA" in c.upper() and "FASE" not in c.upper()][0]

pv_total_5min = (df_inversor[p1].fillna(0) + df_inversor[p2].fillna(0) + df_inversor[p3].fillna(0)).clip(lower=0)
pv_hourly = pv_total_5min.resample("1h").mean().dropna()

df_inversor

,Medida,Inversor 1 - MARCA,Inversor 1 - MODELO,Inversor 1 - ESTADO COM,Inversor 1 - CALIDAD COM,Inversor 1 - NUMERO DE PREGUNTAS,Inversor 1 - NUMERO DE FALLOS,Inversor 1 - FECHA ULTIMA LECTURA,Inversor 1 - DURACION ULTIMO MUESTREO (s),Inversor 1 - ENERGIA ACTIVA (kWh),...,Dispositivo 3 - CORRIENTE DC IN 06 (A),Dispositivo 3 - POTENCIA DC IN 06 (kW),Inversor 3 - HORAS DE FUNCIONAMIENTO (h),Inversor 3 - TEMPERATURA INT (degC),Inversor 3 - RESISTENCIA AISLAMIENTO (Ohm),Inversor 3 - SETPOINT RELATIVO POTENCIA ACTIVA (%),Inversor 3 - SETPOINT RELATIVO POTENCIA REACTIVA (%),Inversor 3 - SMA NUMERO DE SERIE,Inversor 3 - SMA ESTADO,Inversor 3 - SMA CONDICION
Fecha,,,,,,,,,,,,,,,,,,,,,
2025-09-01 00:05:00,0,1.0,0.0,0.0,100.0,-953600.0,NaN,1.756678e+09,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
2025-09-01 00:10:00,1,1.0,0.0,0.0,100.0,3300.0,NaN,1.756678e+09,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
2025-09-01 00:15:00,2,1.0,0.0,0.0,100.0,3350.0,NaN,1.756678e+09,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
2025-09-01 00:20:00,3,1.0,0.0,0.0,100.0,3300.0,NaN,1.756679e+09,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
2025-09-01 00:25:00,4,1.0,0.0,0.0,100.0,3300.0,NaN,1.756679e+09,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-16 21:40:00,19,1.0,0.0,0.0,100.0,3300.0,NaN,1.786909e+09,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
2026-08-16 21:45:00,20,1.0,0.0,0.0,100.0,3350.0,NaN,1.786909e+09,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
2026-08-16 21:50:00,21,1.0,0.0,0.0,100.0,3300.0,NaN,1.786910e+09,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.007526e+09,1467.0,307.0
